# **Sequential Diagnosis with IRIS Agents**
### Reproducing *MAI-DxO* — the MAI Diagnostic Orchestrator

This notebook implements the orchestration scheme from **"Sequential Diagnosis with
Language Models"** ([arXiv:2506.22405](https://arxiv.org/abs/2506.22405), Nori et al.,
Microsoft AI) on top of the **IRIS Agents** framework.

The paper's central idea: rather than answering a static multiple-choice vignette, a
diagnostic AI should behave like a clinician — start from a thin case abstract, then
**iteratively** ask questions and order tests, paying a realistic cost for each, before
committing to a diagnosis. To do this well, the authors wrap a single model in the
**MAI Diagnostic Orchestrator (MAI-DxO)**, which has **one language model role-play a panel
of five physicians** that debate each step and reach consensus.

### The panel (a "chain of debate")

In the paper, **one model role-plays all five personas** in a single deliberation. Here we
materialize each persona as its own IRIS `Agent` — a faithful and more inspectable rendering
of the same roles (you can watch and bill each one separately), differing from the paper only
in that the personas are separate calls rather than one model wearing five hats.

| Paper persona | Responsibility (per the paper) | Materialized here as `Agent` |
|---|---|---|
| **Dr. Hypothesis** | Maintains a probability-ranked **top-3** differential, updated Bayesian-style after each finding | `DrHypothesis` |
| **Dr. Test-Chooser** | Selects up to **5 questions / 3 tests** that best discriminate the leading hypotheses | `DrTestChooser` |
| **Dr. Challenger** | Devil's advocate — flags anchoring bias and proposes falsifying tests | `DrChallenger` |
| **Dr. Stewardship** | Cost-conscious care — vetoes low-yield tests, finds cheaper equivalents | `DrStewardship` |
| **Dr. Checklist** | **Silent quality control** — valid test names, internal consistency (not a decision-maker) | `DrChecklist` |

The panel **reaches consensus** on one of three actions each turn — ask questions, order
tests, or commit to a diagnosis. There is no chairman; the consensus action is what the
`DrChecklist` step records after running its quality checks. Two non-panelists complete the
benchmark:

| Role (not a panelist) | Responsibility | `Agent` |
|---|---|---|
| **Gatekeeper** | Holds the *full* case; reveals only what's explicitly asked, synthesizing case-consistent findings for anything not in the record | `Gatekeeper` |
| **Judge** | Scores the committed diagnosis against ground truth on a **1–5** scale | `Judge` |

### How the framework maps to the paper
Each role is a persisted **`Agent`** with its own **`Prompt`** and a **structured output**
(`response_format`). The orchestration loop runs in Python. The Gatekeeper keeps its memory
of the unfolding case in a persisted **`Chat`**, and token spend across the whole panel is
tracked with **`usage()`** — so the entire multi-agent system, including its telemetry, lives
on InterSystems IRIS.

> **Prerequisites** — same as `demo.ipynb`: IRIS 2025.1+ reachable with the credentials in
> `.env`, and an `OPENAI_API_KEY`. No MCP servers are required for this notebook.

> **On the cases** — the authors did **not** release SDBench; the 304 NEJM CPC cases are
> paywalled and used by permission. This notebook therefore runs on **synthetic, CPC-style
> cases** written for the demo (clearly labelled as such). They reproduce the benchmark's
> *shape* — a public abstract, a sealed full record, and a ground-truth diagnosis — not its
> actual contents.

## 1. Setup

We import the framework and set a few knobs. The paper deliberately uses **cheaper models
for the supporting roles** (Gatekeeper on o4-mini, Judge on o3, orchestrator on GPT-4.1); for
a self-contained demo we use one model everywhere, but `MODEL` is a single switch. The paper
caps each turn at **5 questions and 3 tests**; we mirror those limits. Reasoning effort is
kept modest so the full frontier sweep (every mode × every case) stays affordable — raise it
toward the paper's accuracy-maximizing configuration.

In [1]:
from agents import Agent, Production, Chat, Prompt
from pydantic import BaseModel, Field

# 'gpt' models route to OpenAI inside the framework. The paper is model-agnostic and uses
# cheaper models for the Gatekeeper/Judge; here one model serves every role for simplicity.
MODEL = 'gpt-5'
REASONING_EFFORT = 'low'

# Per-turn action limits, matching the paper (5 questions or 3 tests per turn).
MAX_QUESTIONS_PER_TURN = 5
MAX_TESTS_PER_TURN = 3
MAX_ROUNDS = 4               # the paper allows many turns; we cap it for a quick, cheap demo
CORRECT_THRESHOLD = 4        # paper: a Judge score >= 4 counts as a correct diagnosis

## 2. Structured outputs — one schema per role

The paper's panel only works because each persona produces a *well-formed, machine-readable*
contribution the next stage can consume. In IRIS Agents we express that with Pydantic models
passed as each agent's `response_format`; the framework compiles them into IRIS classes and
enforces them as strict JSON schemas.

`DrChecklist` is **not** a decision-maker in the paper — it is silent quality control. So its
output records the panel's **consensus action** *plus* its QC findings (whether test names are
valid and the plan is internally consistent). The `Verdict` carries only the Judge's 1–5
score; the paper's "correct" threshold (≥4) is applied by us afterward, not by the model.

In [2]:
class Hypothesis(BaseModel):
    diagnosis: str
    probability: float = Field(description="Calibrated likelihood from 0 to 1")
    supporting_evidence: str

class Differential(BaseModel):
    """Dr. Hypothesis — a probability-ranked top-3 differential, updated Bayesian-style."""
    hypotheses: list[Hypothesis]
    summary: str

class ProposedAction(BaseModel):
    kind: str          # 'question' (history/exam) or 'test' (a lab/imaging study)
    detail: str        # the exact question asked, or the test ordered
    rationale: str     # which hypotheses it discriminates

class TestPlan(BaseModel):
    """Dr. Test-Chooser — discriminating next actions (<=5 questions or <=3 tests)."""
    actions: list[ProposedAction]
    reasoning: str

class Challenge(BaseModel):
    """Dr. Challenger — the devil's advocate."""
    biases_identified: str
    alternative_diagnoses: list[str]
    falsifying_tests: list[str]
    critique: str

class StewardshipReview(BaseModel):
    """Dr. Stewardship — cost-conscious gatekeeping of the plan."""
    approved_actions: list[str]
    vetoed_actions: list[str]
    cheaper_alternatives: str
    reasoning: str

class ConsensusAction(BaseModel):
    """Dr. Checklist — silent QC that records the panel's consensus action this turn.

    Checklist does not 'decide' for the panel; it validates the deliberation (real test
    names, internal consistency) and reports the consensus the panel converged on. The
    framework marks every field required, so each is ALWAYS populated: 'requests' is empty
    on a 'diagnose' turn, and 'final_diagnosis' carries the current best guess (committed
    when action == 'diagnose').
    """
    action: str               # 'ask' | 'test' | 'diagnose' (the panel's consensus)
    requests: list[str]       # questions/tests to send to the Gatekeeper; [] when diagnosing
    final_diagnosis: str      # best current guess; the committed answer when diagnosing
    quality_notes: str        # QC: are test names valid and the plan internally consistent?

class GatekeeperFinding(BaseModel):
    """Gatekeeper — discloses only what was explicitly requested."""
    findings: list[str]
    note: str

class Verdict(BaseModel):
    """Judge — 1 to 5 only; the >=4 'correct' threshold is applied by us, not the model."""
    score: int
    rationale: str

## 3. Versioned prompts for each role

Prompts are first-class, versioned objects in IRIS Agents. We author one per role. This makes
the panel's "constitution" auditable and A/B-testable — exactly what you'd want when tuning a
clinical system.

The paper defines several **operating modes** along the cost–accuracy frontier (Instant
Answer, Question-Only, Budgeted, Unconstrained, Ensemble). A full reproduction would add a
separate **budget-tracker** call that estimates each test's cost and can cancel it; here we
approximate the *unconstrained* vs *budgeted* contrast more simply, as two **versions** of the
Dr. Stewardship prompt — a faithful nod to the modes without the full budget-tracker
machinery.

In [3]:
hypothesis_prompt = Prompt(name='dr_hypothesis', text=(
    "You are Dr. Hypothesis, a master diagnostician on a virtual case-conference panel. "
    "Maintain a probability-ranked differential of the THREE most likely diagnoses given "
    "everything revealed so far, each with a calibrated probability (they need not sum to 1). "
    "Update these probabilities in a BAYESIAN manner as each new finding arrives - raising or "
    "lowering each hypothesis in light of the new evidence. Do not anchor prematurely."
))

test_chooser_prompt = Prompt(name='dr_test_chooser', text=(
    "You are Dr. Test-Chooser. Propose the next actions that would most efficiently "
    "DISCRIMINATE between the current leading hypotheses - history/exam questions (up to "
    "FIVE) or diagnostic tests (up to THREE). Prefer high-information-value actions; for "
    "each, state which hypotheses it confirms or rules out. Do not order tests whose result "
    "would not change management."
))

challenger_prompt = Prompt(name='dr_challenger', text=(
    "You are Dr. Challenger, the panel's devil's advocate. Scrutinize the current "
    "differential and proposed plan for cognitive bias - especially anchoring and "
    "confirmation bias. Name plausible alternative diagnoses the panel may be neglecting, "
    "and tests that could FALSIFY the leading hypothesis rather than merely confirm it."
))

# Dr. Stewardship - v1: standard cost-awareness
stewardship_prompt = Prompt(name='dr_stewardship', text=(
    "You are Dr. Stewardship, responsible for cost-conscious, high-value care. Review the "
    "proposed actions. Approve those whose information value justifies their cost; veto "
    "low-yield or redundant tests; suggest cheaper equivalents of equal diagnostic value. "
    "Never compromise patient safety to save money."
))
print("Stewardship prompt is version", stewardship_prompt.version)

Stewardship prompt is version 1


In [4]:
# Dr. Stewardship - v2: the paper's "budgeted" variant (stricter). Re-initializing the
# same name with new text creates version 2 while version 1 is retained.
stewardship_budget = Prompt(name='dr_stewardship', text=(
    "You are Dr. Stewardship operating under a STRICT diagnostic budget. Approve only the "
    "single highest-value action per round unless two are jointly decisive. Aggressively "
    "veto anything that is not immediately decision-changing, and always prefer the cheapest "
    "test that can resolve the leading question. Patient safety still overrides cost."
))
print("Now at version", stewardship_budget.version,
      "| v1 still retrievable:", Prompt('dr_stewardship', version=1).version)

# This demo runs the standard policy (v1); flip the line below to run the budgeted panel.
ACTIVE_STEWARDSHIP = Prompt('dr_stewardship', version=1)

Now at version 2 | v1 still retrievable: 1


In [5]:
checklist_prompt = Prompt(name='dr_checklist', text=(
    "You are Dr. Checklist, the panel's silent quality-control member. You are NOT the "
    "decision-maker - the panel reaches its own consensus. Your job is to (a) verify that "
    "every proposed test name is real and well-formed and that the plan is internally "
    "consistent, recording any issues in 'quality_notes', and (b) record the panel's "
    "consensus action for this turn. ALWAYS populate every field:\n"
    "  - action='ask'      -> put history/exam questions in 'requests'\n"
    "  - action='test'     -> put test names in 'requests'\n"
    "  - action='diagnose' -> set 'requests' to an empty list\n"
    "ALWAYS fill 'final_diagnosis' with the single most likely diagnosis given the evidence "
    "so far (the panel's best current guess on 'ask'/'test' turns; the committed answer on a "
    "'diagnose' turn). The consensus is to 'diagnose' only when a leading hypothesis is well "
    "supported and further testing would not change management; otherwise gather more "
    "information. Honor Dr. Stewardship's vetoes."
))

# A single diagnostic agent (no panel) - the paper's bare-model baseline that MAI-DxO is
# compared against. It plays all the steps itself and emits the same ConsensusAction schema.
# It is a neutral, competent diagnostician with no panel and no cost conscience - faithful to
# the paper's bare-model comparison point.
solo_prompt = Prompt(name='solo_diagnostician', text=(
    "You are an expert diagnostician working a case sequentially. Each turn, decide ONE "
    "action and ALWAYS populate every field:\n"
    "  - action='ask'      -> put up to five history/exam questions in 'requests'\n"
    "  - action='test'     -> put up to three diagnostic tests in 'requests'\n"
    "  - action='diagnose' -> set 'requests' to an empty list\n"
    "ALWAYS fill 'final_diagnosis' with your single best current diagnosis. Gather the "
    "history and tests you need to distinguish the leading possibilities, and commit to "
    "'diagnose' once a leading diagnosis is well supported. Use 'quality_notes' for brief "
    "reasoning."
))

# One Gatekeeper serves every case. The confidential case file is the first message in each
# case's Chat (seeded in the run loop), so the Gatekeeper reads it from the conversation the
# framework reconstructs from IRIS - no per-case agent needed.
gatekeeper_prompt = Prompt(name='gatekeeper', text=(
    "You are the clinical information Gatekeeper for a case-conference simulation. The COMPLETE "
    "case file has been provided to you as the first message in this conversation. A diagnostic "
    "panel will ask history/exam questions and order tests. Disclose ONLY information that is "
    "explicitly requested - never volunteer the diagnosis, an interpretation, or unrequested "
    "findings, and never reveal a pathognomonic result unless the corresponding test was "
    "specifically ordered. If a requested detail is not in the case file, return a REALISTIC "
    "SYNTHETIC finding that is numerically and clinically consistent with the rest of the case, "
    "with NO indication that it is synthetic. Refuse vague or non-specific requests, asking for "
    "a concrete question or test. Be concise and factual."
))

# The Judge sees the ENTIRE case file during adjudication, as in the paper. The case file
# is passed in the message at call time (one Judge serves every case).
judge_prompt = Prompt(name='dx_judge', text=(
    "You are an expert physician judging diagnostic accuracy against a physician-authored "
    "rubric. You are given the full case file, the candidate diagnosis, and the ground-truth "
    "diagnosis. Compare the candidate to the ground truth on the core disease entity, "
    "etiology, anatomic site, and specificity, and assign a single score on this 5-point "
    "Likert scale:\n"
    "  5 - Perfect/clinically superior: identical to, or a strictly more specific version of, "
    "the ground truth, with no unrelated or incorrect additions.\n"
    "  4 - Mostly correct, minor incompleteness: core disease right, a secondary qualifier "
    "missing or slightly mis-specified; management largely unchanged.\n"
    "  3 - Partially correct, major error: right general category but a major error in "
    "etiology, site, or specificity (or an unrelated diagnosis included alongside a correct one).\n"
    "  2 - Largely incorrect: superficial overlap only; fundamentally misdirects the work-up.\n"
    "  1 - Completely incorrect: no meaningful overlap; would be harmful if acted upon.\n"
    "Return only the integer score and your rationale."
))

## 4. The case

The benchmark converts NEJM Clinicopathological Conference (CPC) cases into stepwise
encounters: the solver first sees only a short **abstract**, while the full record stays
behind the Gatekeeper, and a **ground-truth diagnosis** is used by the Judge.

Here we use a **real NEJM CPC case** (Case 3-2026 — a 58-year-old woman with diplopia and
fever), provided scrubbed. The public `abstract` is the presenting complaint; the `full`
record — history, exam, and initial workup — is what the Gatekeeper holds and reveals on
request. `CASES` is just a list, so additional cases drop in with the same
`id` / `abstract` / `full` / `ground_truth` shape.

> **Two fields need authoritative input** (flagged in the cell): the record was scrubbed of
> its diagnosis, so `ground_truth` must be the published NEJM answer (the Judge scores against
> it), and ideally `full` should include the confirmatory results (serology/CSF/biopsy) so the
> Gatekeeper discloses *real* findings when the right test is ordered. With only the scrubbed
> text, the Gatekeeper synthesizes those results and the run is illustrative rather than exact.

Accuracy and cost are still computed the paper's way — **score ≥4 = correct** and **\$/case** —
but with a single case the "frontier" is one point per mode; add more cases for a meaningful
curve.

In [6]:
# Real NEJM Clinicopathological Conference case (Case 3-2026), provided scrubbed: the public
# `abstract` is the brief presenting complaint, and `full` is the scrubbed case record the
# Gatekeeper holds. `ground_truth` is the published NEJM final diagnosis (the Judge scores
# against it).
#
# NOTE on fidelity: the scrubbed record stops at the initial workup - it contains no
# confirmatory results (Lyme/Babesia serology, smear, PCR, CSF). So when the panel orders the
# right test the Gatekeeper has no real result to disclose and will SYNTHESIZE a plausible,
# case-consistent finding (per its prompt). That synthetic finding may not point at the true
# coinfection, so treat this single-case run as illustrative. Appending the real confirmatory
# results to `full` would make it exact.

CASES = [
    {
        'id': 'diplopia-fever',
        'abstract': (
            "A 58-year-old woman was admitted because of double vision and fever. Three weeks "
            "earlier she had undergone lumbar disk surgery; afterward she had persistent left "
            "leg pain, and in the week before admission she developed a spreading rash, "
            "recurrent fevers, a swollen painful right eyelid, and horizontal diplopia."
        ),
        'full': """Case 3-2026: A 58-Year-Old Woman with Diplopia and Fever

A 58-year-old woman was admitted to this hospital because of diplopia and fever. Three weeks
before the current presentation, the patient underwent lumbar hemilaminectomy, facetectomy,
and foraminotomy on the left side, with excision of a herniated disk and decompression of the
nerve root, at another hospital. On postoperative day 1, back pain increased substantially,
and oxycodone was prescribed. Owing to the back pain, which did not abate after oxycodone
treatment, and the development of left leg pain with resultant limited mobility, prednisone
was prescribed on postoperative day 3.

On postoperative day 4, the patient was readmitted to the other hospital. The blood levels of
glucose and electrolytes were normal, as were the results of tests of kidney function. Magnetic
resonance imaging (MRI) of the lumbar spine, performed after the administration of intravenous
contrast material, reportedly showed postsurgical changes and an epidural fluid collection,
measuring 16 mm in diameter, on the left side at the level of the fifth lumbar vertebra; a
radiograph of the left leg was reportedly normal, and ultrasonography showed no deep venous
thrombosis in the legs. Treatment with intravenous dexamethasone and oral acetaminophen,
cyclobenzaprine, celecoxib, gabapentin, hydromorphone, and tramadol was administered.

Seven days before the current presentation, leg pain persisted, and the patient noticed a new
erythematous rash on her legs, chest, buttocks, back, and cheeks. She contacted her primary
care physician and reported that the rash had started as hives and had then become "welts." The
tympanic temperature measured by the patient at home was 39.1 C, and she was instructed by her
primary care physician to take diphenhydramine and ibuprofen and to stop taking all other
medications except gabapentin.

Six days later, the patient was evaluated by her primary care physician, who prescribed oral
doxycycline for a presumed skin infection. The next day, fever recurred, the right eyelid became
swollen and painful with motion, and horizontal diplopia developed. The patient presented to the
emergency department of this hospital.

On evaluation in the emergency department, the patient reported ongoing leg pain and fever but
no chills or diaphoresis, although she recalled that she had had 3 days of chills and body aches
before the lumbar surgery. Aside from diplopia, which decreased with covering one eye, she did
not report worsening visual acuity or changes in color vision. She had no hearing loss, sinus or
ear symptoms, headache, neck stiffness, weakness, falls, or paresthesia (except for chronic
numbness in the toes of the left foot). She reported a persistent erythematous rash on her legs.
She had no weight change, photosensitivity associated with the rash, oral or genital ulcers,
Raynaud's phenomenon, jaw claudication, dysphagia, lightheadedness, dyspnea, or chest pain. No
pulmonary, gastroenterologic, or genitourinary symptoms were present.

Medical history included lumbar spondylosis and disk protrusion with radiculopathy, type 2
diabetes mellitus, hypertension, hyperlipidemia, coronary artery disease, colonic polyps,
obstructive sleep apnea, asthma, allergic rhinitis, eczema, anxiety, and periodic limb movement
disorder. Medications included gabapentin, aspirin, rosuvastatin, losartan, hydrochlorothiazide,
and sertraline, as well as clonazepam as needed. Amoxicillin had caused lip swelling, and
lisinopril had caused a cough. The patient had received vaccinations against SARS-CoV-2.

The patient worked in education and lived in a coastal suburban area of eastern Massachusetts
with her husband and dog. She frequently walked her dog in a wooded area. Recent travel included
a trip to Europe; she had had no specific animal, food, or insect exposures. She occasionally
drank wine and did not use tobacco or other substances. Her family history included
hyperlipidemia and coronary artery disease in her father, mother, and brother. Her father also
had diabetes and diverticulosis, and her maternal grandmother had had breast cancer.

On examination, the temporal temperature was 36.1 C, the heart rate 80 beats per minute, the
blood pressure 125/80 mm Hg, and the oxygen saturation 97% while breathing ambient air. The
body-mass index was 34.4. Ophthalmologic examination revealed right esotropia, which was worse
with attempted abduction, and slight ptosis, with the right pupil slightly smaller than the left
pupil. No conjunctival erythema or scleral icterus was present. The remainder of the neurologic
examination, including evaluation of other cranial-nerve function, strength, sensation,
reflexes, and coordination, was normal. The lumbar surgical-incision site was well-approximated
and clean, without drainage or fluctuance. Confluent erythematous macules, measuring
approximately 5 cm in diameter overall, were present on the right lower leg and buttocks,
without palpable purpura. There was no palpable lymphadenopathy. The remainder of the
examination was normal.

The blood levels of electrolytes, glucose, albumin, globulin, aspartate aminotransferase,
alanine aminotransferase, alkaline phosphatase, and creatine kinase were normal, as were the
white-cell count and differential count and the results of tests of kidney function. Tests for
SARS-CoV-2, influenza A and B viruses, human immunodeficiency virus types 1 and 2, and hepatitis
C virus antibodies were negative. Urinalysis was normal.

A chest radiograph was normal. Computed tomographic (CT) angiography of the head and neck,
performed after the administration of intravenous contrast material, showed no intracranial
vascular abnormality. MRI of the head, performed after the administration of intravenous
contrast material, showed no infarct or mass, but enhancement was noted along cranial nerves
III, V, VI, and VII on the right side.

Treatment with intravenous acyclovir and oral doxycycline, acetaminophen, ibuprofen, and
diazepam was administered. The patient was admitted to the neurology unit of this hospital.""",
        'ground_truth': "Lyme neuroborreliosis with coinfection with Babesia microti.",
    },
]

for case in CASES:
    print(f"[{case['id']}] {case['abstract'][:90]}...")

[diplopia-fever] A 58-year-old woman was admitted because of double vision and fever. Three weeks earlier s...


## 5. The cost model — the paper's second axis

MAI-DxO is scored on accuracy **and** cost. The paper prices an encounter two ways:

- **Physician visit** — a flat **\$300** that covers a *batch* of sequential history/exam
  questions. Individual questions are **not** billed separately; a visit "concludes upon a
  diagnostic test request." So an `ask` turn costs one \$300 visit regardless of how many
  questions it bundles.
- **Tests** — priced individually. The paper maps each free-text test to **CPT codes** and a
  2023 US price table. We can't ship that table, so we approximate with a small CPT-style
  catalog and a default for unmatched tests; in a real IRIS deployment this catalog would live
  in an IRIS table the panel queries through a Toolkit.

Total case cost = visit fees + test costs incurred before the diagnosis is committed.

In [7]:
VISIT_COST = 300          # flat fee for one physician visit (a batch of questions)

# CPT-style price catalog (USD), approximating the paper's CPT -> 2023 price-table mapping.
# A test is priced by the first keyword it matches; unmatched tests fall back to the default.
TEST_PRICES = {
    'metanephrine': 250, 'catecholamine': 220, 'tsh': 40, 't4': 40, 'calcitonin': 90,
    'calcium': 20, 'glucose': 15, 'cbc': 30, 'electrolyte': 30, 'renin': 110,
    'aldosterone': 110, 'ct': 600, 'mri': 1200, 'mibg': 1500, 'genetic': 400, 'ret': 400,
    'ultrasound': 250, 'echocardiogram': 450, 'ecg': 60, 'ekg': 60, 'x-ray': 90,
    'xray': 90, 'culture': 110, 'biopsy': 800, 'lumbar': 350, 'csf': 350, 'pcr': 180,
    'antibody': 130, 'serology': 130, 'lactate': 35, 'ammonia': 60, 'cortisol': 70,
    'ferritin': 45, 'esr': 25, 'crp': 30, 'ldh': 25, 'urinalysis': 20, 'toxicology': 200,
    'osmolality': 30, 'flow cytometry': 600, 'pet': 2000, 'endoscopy': 900, 'colonoscopy': 1000,
}
DEFAULT_TEST_COST = 150   # unmatched test -> paper LM-estimates; we use a flat default

# A single test's price, used inline where tests are billed:
#   next((c for k, c in TEST_PRICES.items() if k in test.lower()), DEFAULT_TEST_COST)

## 6. Instantiate the panel and start the Production

Each role becomes an `Agent` bound to its prompt, model, and structured output. There is **one
Gatekeeper** for all cases: rather than baking a case file into its system prompt, we put the
confidential record as the **first message of each case's `Chat`** (seeded in the run loop), so
the Gatekeeper reads it from the conversation the framework reconstructs from IRIS — the same
way the Judge receives the file in its message. The panel agents are separate calls and never
see that chat, so the case stays sealed.

All agents are registered in a single `Production` and started; afterwards any can be called
independently.

In [8]:
dr_hypothesis  = Agent(name='DrHypothesis', 
                       model=MODEL, 
                       system_prompt=hypothesis_prompt,
                       response_format=Differential, 
                       reasoning_effort=REASONING_EFFORT)
dr_testchooser = Agent(name='DrTestChooser', 
                       model=MODEL, 
                       system_prompt=test_chooser_prompt,
                       response_format=TestPlan, 
                       reasoning_effort=REASONING_EFFORT)
dr_challenger  = Agent(name='DrChallenger', 
                       model=MODEL, 
                       system_prompt=challenger_prompt,
                       response_format=Challenge, 
                       reasoning_effort=REASONING_EFFORT)
dr_stewardship = Agent(name='DrStewardship', 
                       model=MODEL, 
                       system_prompt=ACTIVE_STEWARDSHIP,
                       response_format=StewardshipReview, 
                       reasoning_effort=REASONING_EFFORT)
dr_checklist   = Agent(name='DrChecklist', 
                       model=MODEL, 
                       system_prompt=checklist_prompt,
                       response_format=ConsensusAction, 
                       reasoning_effort=REASONING_EFFORT)
solo_doctor    = Agent(name='SoloDiagnostician',     # bare-model baseline (no panel)
                       model=MODEL,
                       system_prompt=solo_prompt,
                       response_format=ConsensusAction,
                       reasoning_effort=REASONING_EFFORT)
gatekeeper     = Agent(name='Gatekeeper',            # one Gatekeeper; case file lives in each case's Chat
                       model=MODEL,
                       system_prompt=gatekeeper_prompt,
                       response_format=GatekeeperFinding,
                       reasoning_effort=REASONING_EFFORT)
judge          = Agent(name='Judge', 
                       model=MODEL, 
                       system_prompt=judge_prompt,
                       response_format=Verdict, 
                       reasoning_effort=REASONING_EFFORT)

panel = [dr_hypothesis, dr_testchooser, dr_challenger, dr_stewardship, dr_checklist]
all_agents = panel + [solo_doctor, gatekeeper, judge]

Production(name='MAIDxO', agents=all_agents).start()


Load started on 06/12/2026 12:28:21
Loading file Agents.Message.Hypothesis.cls as udl
Compiling class Agents.Message.Hypothesis
Compiling routine Agents.Message.Hypothesis.1
Load finished successfully.

Load started on 06/12/2026 12:28:22
Loading file Agents.Message.Differential.cls as udl
Compiling class Agents.Message.Differential
Compiling table Agents_Message.Differential
Compiling routine Agents.Message.Differential.1
Load finished successfully.

Load started on 06/12/2026 12:28:22
Loading file Agents.Message.ProposedAction.cls as udl
Compiling class Agents.Message.ProposedAction
Compiling routine Agents.Message.ProposedAction.1
Load finished successfully.

Load started on 06/12/2026 12:28:22
Loading file Agents.Message.TestPlan.cls as udl
Compiling class Agents.Message.TestPlan
Compiling table Agents_Message.TestPlan
Compiling routine Agents.Message.TestPlan.1
Load finished successfully.

Load started on 06/12/2026 12:28:23
Loading file Agents.Message.Challenge.cls as udl
Compil

## 7. Operating modes

The paper doesn't run one configuration — it places several **operating modes** on the
cost–accuracy frontier and compares them. The next four sections each run one mode, end to
end, over all three cases:

| Section | Mode | Engine | Gathering | Cost |
|---|---|---|---|---|
| 8 | Instant Answer | one solo model, abstract only | none | \$0 |
| 9 | No Panel | one solo model, sequential | ask + test | visits + tests |
| 10 | Question-Only | full 5-persona panel | questions only | visits only |
| 11 | Unconstrained Panel | full 5-persona panel | ask + test | visits + tests |

`No Panel` is the paper's **bare-model baseline** that MAI-DxO is measured against;
`Unconstrained Panel` is the full orchestrator. Every mode ends the same way: the **Judge**,
given the full case file, scores the committed diagnosis 1–5, and we record `(score, cost)`.
Each mode appends its rows to a shared `results` list that Section 12 turns into the frontier.

In [9]:
# Every mode appends {mode, case, final_diagnosis, score, correct, cost} rows here.
results = []

## 8. Instant Answer

The cheapest mode and the paper's floor: the solo model sees **only the public abstract** and
must commit to a diagnosis immediately — no questions, no tests, **\$0**. It shows how far a
thin vignette alone gets you.

In [10]:
for case in CASES:
    print(f"\n{'='*72}\nINSTANT ANSWER  —  {case['id']}\n{'='*72}")

    # Diagnose straight from the abstract; force a commitment, no gathering.
    decision = solo_doctor(
        f"Case:\n{case['abstract']}\n\n"
        "No further information is available. You MUST choose action='diagnose' now.")
    decision = ConsensusAction.model_validate_json(decision)
    final_diagnosis = decision.final_diagnosis or "undetermined"

    print("  Diagnostician  →  DIAGNOSE")
    print(f"      reasoning: {decision.quality_notes}")
    print(f"      diagnosis: {final_diagnosis}")

    # Judge sees the full case file, as in the paper; we apply the >=4 'correct' rule.
    verdict = judge(
        f"===== FULL CASE FILE =====\n{case['full']}\n\n"
        f"Ground-truth diagnosis: {case['ground_truth']}\n"
        f"Candidate diagnosis: {final_diagnosis}\n"
        "Score the candidate against the ground truth.")
    verdict = Verdict.model_validate_json(verdict)
    correct = verdict.score >= CORRECT_THRESHOLD

    results.append({'mode': 'instant', 'case': case['id'], 'final_diagnosis': final_diagnosis,
                    'score': verdict.score, 'correct': correct, 'cost': 0})
    print(f"\n  JUDGE: {verdict.score}/5 ({'correct' if correct else 'incorrect'})  |  cost $0")


INSTANT ANSWER  —  diplopia-fever
  Diagnostician  →  DIAGNOSE
      reasoning: Fever, painful swollen eyelid, and horizontal diplopia are classic for cavernous sinus involvement with abducens palsy. Onset after spreading facial/periorbital infection fits septic cavernous sinus thrombosis, a post-septic complication often due to Staphylococcus aureus. This best unifies fever, eyelid edema, and isolated horizontal diplopia.
      diagnosis: Septic cavernous sinus thrombosis (likely from facial/orbital cellulitis) causing CN VI palsy

  JUDGE: 2/5 (incorrect)  |  cost $0


## 9. No Panel — the bare-model baseline

A single diagnostician runs the sequential loop on its own: each turn it decides whether to
ask, test, or diagnose, and the Gatekeeper responds. No debate, no personas. This is the
baseline the paper compares MAI-DxO against — the same task and cost model, minus the panel.

In [11]:
for case in CASES:
    print(f"\n{'='*72}\nNO PANEL  —  {case['id']}\n{'='*72}")

    # Seal the confidential case file as the Gatekeeper's case memory for this case.
    case_chat = Chat(name=f"sdbench-{case['id']}-no_panel", messages=[
        {'role': 'assistant', 'content': "CONFIDENTIAL CASE FILE:\n" + case['full']}])
    revealed = "INITIAL ABSTRACT:\n" + case['abstract']
    cost = 0
    final_diagnosis = "undetermined"

    for round in range(1, MAX_ROUNDS + 1):
        is_last = (round == MAX_ROUNDS)
        print(f"\n  ── Round {round} ──")

        # One solo model decides the action - no panel deliberation.
        raw = solo_doctor(
            f"Case so far:\n{revealed}\n\n"
            + ("This is the FINAL round - you MUST choose action='diagnose'.\n" if is_last else "")
            + "Decide this turn's single action.")
        decision = ConsensusAction.model_validate_json(raw)
        action = 'diagnose' if is_last else decision.action.strip().lower()
        print(f"  Diagnostician  →  {action.upper()}")
        print(f"      reasoning: {decision.quality_notes}")

        if action == 'diagnose':
            final_diagnosis = decision.final_diagnosis or "undetermined"
            print(f"      diagnosis: {final_diagnosis}")
            break

        if action == 'ask':
            requests = decision.requests[:MAX_QUESTIONS_PER_TURN]
            cost += VISIT_COST                                  # flat $300 per visit
        else:  # 'test'
            requests = decision.requests[:MAX_TESTS_PER_TURN]
            for test in requests:
                cost += next((price for keyword, price in TEST_PRICES.items()
                              if keyword in test.lower()), DEFAULT_TEST_COST)
        for request in requests:
            print(f"      • {request}")

        raw = gatekeeper("The diagnostician requests the following. Disclose only what is asked:\n- "
                         + "\n- ".join(requests), chat=case_chat)
        finding = GatekeeperFinding.model_validate_json(raw)
        print("  Gatekeeper  →  reveals")
        for found in finding.findings:
            print(f"      • {found}")
        revealed += f"\n\nROUND {round} ({action}) requested {requests}\nFindings: {finding.findings}"

    raw = judge(
        f"===== FULL CASE FILE =====\n{case['full']}\n\n"
        f"Ground-truth diagnosis: {case['ground_truth']}\n"
        f"Candidate diagnosis: {final_diagnosis}\n"
        "Score the candidate against the ground truth.")
    verdict = Verdict.model_validate_json(raw)
    correct = verdict.score >= CORRECT_THRESHOLD

    results.append({'mode': 'no_panel', 'case': case['id'], 'final_diagnosis': final_diagnosis,
                    'score': verdict.score, 'correct': correct, 'cost': cost})
    print(f"\n  JUDGE: {verdict.score}/5 ({'correct' if correct else 'incorrect'})  |  cost ${cost}")


NO PANEL  —  diplopia-fever

  ── Round 1 ──
  Diagnostician  →  ASK
      reasoning: Fever, painful swollen eyelid, and new horizontal diplopia (likely CN VI palsy) after recent surgery raise concern for orbital infection with spread to cavernous sinus. Need features of orbital involvement, sinus source, meningitic symptoms, and postop infection source.
      • Any eye pain with movement, decreased vision, or proptosis on the right?
      • Are there recent sinus/nasal symptoms (congestion, purulent discharge, facial pain) or dental infection?
      • Describe the rash (morphology, distribution, tenderness) and whether it is petechial, vesicular, or erythematous; any mucosal involvement?
      • Any severe headache, neck stiffness, photophobia, or altered mental status?
      • Any wound issues after the lumbar surgery (redness, drainage), persistent back pain with fever, or bacteremia signs?
  Gatekeeper  →  reveals
      • Right eye: pain with movement present; no decreased vision;

## 10. Question-Only Panel

The full panel deliberates, but **tests are off the table** — it may only ask history/exam
questions or commit. Cost is therefore visits only (no test charges). This isolates how much
of the panel's accuracy comes from clinical reasoning over history versus from ordering tests.

In [12]:
for case in CASES:
    print(f"\n{'='*72}\nQUESTION-ONLY PANEL  —  {case['id']}\n{'='*72}")

    # Seal the confidential case file as the Gatekeeper's case memory for this case.
    case_chat = Chat(name=f"sdbench-{case['id']}-question_only", messages=[
        {'role': 'assistant', 'content': "CONFIDENTIAL CASE FILE:\n" + case['full']}])
    revealed = "INITIAL ABSTRACT:\n" + case['abstract']
    cost = 0
    final_diagnosis = "undetermined"

    for round in range(1, MAX_ROUNDS + 1):
        is_last = (round == MAX_ROUNDS)
        print(f"\n  ── Round {round} ──")

        # The chain of debate, same as the full panel, but tests are off the table.
        raw = dr_hypothesis(f"Case so far:\n{revealed}\n\nGive your ranked differential.")
        differential = Differential.model_validate_json(raw)
        print("  Dr. Hypothesis  →  differential")
        for hypothesis in differential.hypotheses:
            print(f"      • {hypothesis.diagnosis}  (p={hypothesis.probability:.2f})")

        raw = dr_testchooser(
            f"Case so far:\n{revealed}\n\nLeading differential:\n{differential.summary}\n"
            f"Tests are NOT available; propose at most {MAX_QUESTIONS_PER_TURN} history/exam questions.")
        plan = TestPlan.model_validate_json(raw)
        print("  Dr. Test-Chooser  →  proposes questions")
        for proposed in plan.actions:
            print(f"      • {proposed.detail}")

        raw = dr_challenger(
            f"Case so far:\n{revealed}\n\nDifferential:\n{differential.summary}\n"
            f"Proposed questions:\n{[a.detail for a in plan.actions]}\nChallenge this reasoning.")
        challenge = Challenge.model_validate_json(raw)
        print(f"  Dr. Challenger  →  {challenge.critique}")
        if challenge.alternative_diagnoses:
            print(f"      alternatives: {', '.join(challenge.alternative_diagnoses)}")

        raw = dr_stewardship(
            f"Proposed questions: {[a.detail for a in plan.actions]}\n"
            f"Challenger's concerns: {challenge.critique}\nReview for value.")
        stewardship = StewardshipReview.model_validate_json(raw)
        print(f"  Dr. Stewardship  →  approved {stewardship.approved_actions}, "
              f"vetoed {stewardship.vetoed_actions}")

        raw = dr_checklist(
            f"Case so far:\n{revealed}\n\n"
            f"Dr. Hypothesis: {differential.summary}\n"
            f"Dr. Test-Chooser proposed: {[a.detail for a in plan.actions]}\n"
            f"Dr. Challenger critique: {challenge.critique}\n"
            f"Dr. Stewardship approved {stewardship.approved_actions}, vetoed {stewardship.vetoed_actions}.\n"
            "Tests are NOT available - the consensus may only be 'ask' or 'diagnose'.\n"
            + ("This is the FINAL round - the consensus MUST be action='diagnose'.\n" if is_last else "")
            + "Run your QC checks and record the panel's consensus action.")
        consensus = ConsensusAction.model_validate_json(raw)
        action = 'diagnose' if is_last else consensus.action.strip().lower()
        if action == 'test':                       # tests disallowed in this mode
            action = 'ask'
        print(f"  Dr. Checklist  →  CONSENSUS: {action.upper()}  (QC: {consensus.quality_notes})")

        if action == 'diagnose':
            final_diagnosis = consensus.final_diagnosis or "undetermined"
            print(f"      diagnosis: {final_diagnosis}")
            break

        requests = consensus.requests[:MAX_QUESTIONS_PER_TURN]
        cost += VISIT_COST                          # questions only -> one $300 visit per turn
        raw = gatekeeper("The panel requests the following. Disclose only what is asked:\n- "
                         + "\n- ".join(requests), chat=case_chat)
        finding = GatekeeperFinding.model_validate_json(raw)
        print("  Gatekeeper  →  reveals")
        for found in finding.findings:
            print(f"      • {found}")
        revealed += f"\n\nROUND {round} (ask) requested {requests}\nFindings: {finding.findings}"

    raw = judge(
        f"===== FULL CASE FILE =====\n{case['full']}\n\n"
        f"Ground-truth diagnosis: {case['ground_truth']}\n"
        f"Candidate diagnosis: {final_diagnosis}\n"
        "Score the candidate against the ground truth.")
    verdict = Verdict.model_validate_json(raw)
    correct = verdict.score >= CORRECT_THRESHOLD

    results.append({'mode': 'question_only', 'case': case['id'], 'final_diagnosis': final_diagnosis,
                    'score': verdict.score, 'correct': correct, 'cost': cost})
    print(f"\n  JUDGE: {verdict.score}/5 ({'correct' if correct else 'incorrect'})  |  cost ${cost}")


QUESTION-ONLY PANEL  —  diplopia-fever

  ── Round 1 ──
  Dr. Hypothesis  →  differential
      • Septic cavernous sinus thrombosis (often secondary to sinusitis/orbital infection)  (p=0.45)
      • Orbital cellulitis with extraocular muscle/nerve involvement  (p=0.35)
      • Herpes zoster ophthalmicus with cranial neuropathy  (p=0.15)
  Dr. Test-Chooser  →  proposes questions
      • Check for bilateral ocular involvement: ask and examine for pain/swelling, chemosis/proptosis, and motility limitation in the contralateral eye, and whether diplopia involves both gaze directions or both eyes.
      • Focused cranial nerve exam: III, IV, VI (ptosis, EOM deficits, diplopia patterns, pupil involvement), plus V1/V2 facial sensation and corneal reflex.
      • Inspect the rash closely: morphology (vesicular vs maculopapular/erysipelas-like), crusting, and strict unilateral V1 dermatomal distribution; check tip of nose (Hutchinson sign).
      • Detailed ocular exam: degree of proptosis and 

## 11. Unconstrained Panel — the full MAI-DxO

The full orchestrator: every round runs the complete **chain of debate** — Dr. Hypothesis →
Dr. Test-Chooser → Dr. Challenger → Dr. Stewardship → Dr. Checklist (QC + consensus) — and may
ask, test, or diagnose. This is the configuration the paper's headline accuracy comes from.

In [13]:
for case in CASES:
    print(f"\n{'='*72}\nUNCONSTRAINED PANEL  —  {case['id']}\n{'='*72}")

    # Seal the confidential case file as the Gatekeeper's case memory for this case.
    case_chat = Chat(name=f"sdbench-{case['id']}-unconstrained", messages=[
        {'role': 'assistant', 'content': "CONFIDENTIAL CASE FILE:\n" + case['full']}])
    revealed = "INITIAL ABSTRACT:\n" + case['abstract']
    cost = 0
    final_diagnosis = "undetermined"

    for round in range(1, MAX_ROUNDS + 1):
        is_last = (round == MAX_ROUNDS)
        print(f"\n  ── Round {round} ──")

        # The full chain of debate: Hypothesis -> Test-Chooser -> Challenger -> Stewardship.
        raw = dr_hypothesis(f"Case so far:\n{revealed}\n\nGive your ranked differential.")
        differential = Differential.model_validate_json(raw)
        print("  Dr. Hypothesis  →  differential")
        for hypothesis in differential.hypotheses:
            print(f"      • {hypothesis.diagnosis}  (p={hypothesis.probability:.2f})")

        raw = dr_testchooser(
            f"Case so far:\n{revealed}\n\nLeading differential:\n{differential.summary}\n"
            f"Propose at most {MAX_QUESTIONS_PER_TURN} questions OR {MAX_TESTS_PER_TURN} tests.")
        plan = TestPlan.model_validate_json(raw)
        print("  Dr. Test-Chooser  →  proposes")
        for proposed in plan.actions:
            print(f"      • ({proposed.kind}) {proposed.detail}")

        raw = dr_challenger(
            f"Case so far:\n{revealed}\n\nDifferential:\n{differential.summary}\n"
            f"Proposed plan:\n{[a.detail for a in plan.actions]}\nChallenge this reasoning.")
        challenge = Challenge.model_validate_json(raw)
        print(f"  Dr. Challenger  →  {challenge.critique}")
        if challenge.alternative_diagnoses:
            print(f"      alternatives: {', '.join(challenge.alternative_diagnoses)}")

        raw = dr_stewardship(
            f"Proposed actions: {[a.detail for a in plan.actions]}\n"
            f"Challenger's falsifying tests: {challenge.falsifying_tests}\nReview for cost and value.")
        stewardship = StewardshipReview.model_validate_json(raw)
        print(f"  Dr. Stewardship  →  approved {stewardship.approved_actions}, "
              f"vetoed {stewardship.vetoed_actions}")

        # Dr. Checklist runs QC and records the panel's consensus action.
        raw = dr_checklist(
            f"Case so far:\n{revealed}\n\n"
            f"Dr. Hypothesis: {differential.summary}\n"
            f"Dr. Test-Chooser proposed: {[a.detail for a in plan.actions]}\n"
            f"Dr. Challenger critique: {challenge.critique}\n"
            f"Dr. Stewardship approved {stewardship.approved_actions}, vetoed {stewardship.vetoed_actions}.\n"
            + ("This is the FINAL round - the consensus MUST be action='diagnose'.\n" if is_last else "")
            + "Run your QC checks and record the panel's consensus action.")
        consensus = ConsensusAction.model_validate_json(raw)
        action = 'diagnose' if is_last else consensus.action.strip().lower()
        print(f"  Dr. Checklist  →  CONSENSUS: {action.upper()}  (QC: {consensus.quality_notes})")

        if action == 'diagnose':
            final_diagnosis = consensus.final_diagnosis or "undetermined"
            print(f"      diagnosis: {final_diagnosis}")
            break

        if action == 'ask':
            requests = consensus.requests[:MAX_QUESTIONS_PER_TURN]
            cost += VISIT_COST                                  # flat $300 per visit
        else:  # 'test'
            requests = consensus.requests[:MAX_TESTS_PER_TURN]
            for test in requests:
                cost += next((price for keyword, price in TEST_PRICES.items()
                              if keyword in test.lower()), DEFAULT_TEST_COST)
        for request in requests:
            print(f"      ↳ requesting: {request}")

        raw = gatekeeper("The panel requests the following. Disclose only what is asked:\n- "
                         + "\n- ".join(requests), chat=case_chat)
        finding = GatekeeperFinding.model_validate_json(raw)
        print("  Gatekeeper  →  reveals")
        for found in finding.findings:
            print(f"      • {found}")
        revealed += f"\n\nROUND {round} ({action}) requested {requests}\nFindings: {finding.findings}"

    raw = judge(
        f"===== FULL CASE FILE =====\n{case['full']}\n\n"
        f"Ground-truth diagnosis: {case['ground_truth']}\n"
        f"Candidate diagnosis: {final_diagnosis}\n"
        "Score the candidate against the ground truth.")
    verdict = Verdict.model_validate_json(raw)
    correct = verdict.score >= CORRECT_THRESHOLD

    results.append({'mode': 'unconstrained', 'case': case['id'], 'final_diagnosis': final_diagnosis,
                    'score': verdict.score, 'correct': correct, 'cost': cost})
    print(f"\n  JUDGE: {verdict.score}/5 ({'correct' if correct else 'incorrect'})  |  cost ${cost}")


UNCONSTRAINED PANEL  —  diplopia-fever

  ── Round 1 ──
  Dr. Hypothesis  →  differential
      • Cavernous sinus thrombosis (septic)  (p=0.45)
      • Orbital cellulitis with possible superior ophthalmic vein involvement  (p=0.35)
      • Herpes zoster ophthalmicus with cranial neuropathy  (p=0.20)
  Dr. Test-Chooser  →  proposes
      • (History/Exam) Examine/ask if the rash is vesicular in the V1 (ophthalmic) dermatome, especially nose tip (Hutchinson sign), with dermatomal pain preceding eruption.
      • (Exam) Assess for proptosis, chemosis, and pattern of ophthalmoplegia including CN VI palsy; check visual acuity and relative afferent pupillary defect (RAPD).
      • (History/Exam) Ask about and examine for sinus disease: severe facial pressure, purulent nasal discharge, tenderness over ethmoid/frontal sinuses; look for dental infection.
      • (History/Exam) Determine laterality/evolution: any contralateral ocular signs or new cranial neuropathies developing over hours–days.


## 12. The frontier — accuracy vs. cost per mode

The paper's headline result is not one number but a **trade-off curve**: each operating mode
is a point in (accuracy, mean-cost) space, and the interesting modes form a Pareto frontier.
We aggregate the `results` rows the way the paper does — **accuracy = fraction of cases scored
≥4**, **cost = mean \$/case** — for each mode, so they line up from cheap-and-blind (`instant`,
\$0) toward the information-gathering panel.

(With three cases, accuracy is quantized to 0/33/67/100% — the point is the *shape* of the
comparison, not statistically meaningful values.)

In [14]:
MODES = ['instant', 'no_panel', 'question_only', 'unconstrained']

# Per-(mode, case) detail.
print(f"{'MODE':<16}{'CASE':<18}{'SCORE':>6}{'CORRECT':>9}{'COST':>9}")
print("-" * 58)
for row in results:
    print(f"{row['mode']:<16}{row['case']:<18}{row['score']:>5}/5"
          f"{'yes' if row['correct'] else 'no':>9}{'$'+str(row['cost']):>9}")

# Frontier: one (accuracy, mean cost) point per mode.
print("\n" + "=" * 58)
print(f"{'FRONTIER':<16}{'ACCURACY (>=4)':>18}{'MEAN $/CASE':>14}")
print("=" * 58)
for mode in MODES:
    mode_rows = [row for row in results if row['mode'] == mode]
    accuracy = sum(row['correct'] for row in mode_rows) / len(mode_rows)
    mean_cost = sum(row['cost'] for row in mode_rows) / len(mode_rows)
    print(f"{mode:<16}{accuracy:>17.0%}{'$'+format(int(mean_cost), ','):>14}")

MODE            CASE               SCORE  CORRECT     COST
----------------------------------------------------------
instant         diplopia-fever        2/5       no       $0
no_panel        diplopia-fever        2/5       no    $1830
question_only   diplopia-fever        2/5       no     $900
unconstrained   diplopia-fever        4/5      yes    $1860

FRONTIER            ACCURACY (>=4)   MEAN $/CASE
instant                        0%            $0
no_panel                       0%        $1,830
question_only                  0%          $900
unconstrained                100%        $1,860


## 13. Observability — token spend across the whole panel

Because every agent is a Business Process on IRIS, token usage is logged at each LLM call.
The paper trades tokens for accuracy; IRIS Agents lets us see exactly where they went —
aggregated for the whole `Production`, and broken down per role across every mode and case.

In [15]:
prod = Production('MAIDxO')
print("Whole-production usage:", prod.usage())
print()
for agent in all_agents:
    usage = agent.usage()
    print(f"   {agent.name:<24} total_tokens={usage['total_tokens']:>7}  "
          f"(reasoning={usage['output_reasoning_tokens']})")

Whole-production usage: {'input_tokens': 766614, 'output_tokens': 740404, 'output_reasoning_tokens': 425664, 'total_tokens': 1507018}

   DrHypothesis             total_tokens= 171358  (reasoning=57856)
   DrTestChooser            total_tokens= 215961  (reasoning=86976)
   DrChallenger             total_tokens= 272317  (reasoning=82176)
   DrStewardship            total_tokens= 175186  (reasoning=49344)
   DrChecklist              total_tokens= 204803  (reasoning=47488)
   SoloDiagnostician        total_tokens= 114923  (reasoning=34240)
   Gatekeeper               total_tokens= 141846  (reasoning=26048)
   Judge                    total_tokens=  89672  (reasoning=13312)


In [16]:
# Optional cleanup - removes the production and all agents from IRIS. Uncomment to run.
# Production('MAIDxO').delete()
# for agent in all_agents:
#     agent.delete()